In [17]:
import os
print('Notebook location: ', os.getcwd())
print('CSVs found:', len(os.listdir('./data/raw/')))

Notebook location:  /Users/chauanh/GitHub/olist_commercial_analytics
CSVs found: 9


In [18]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10,6)

In [19]:
data_path = './data/raw/'

order_date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

customers = pd.read_csv(data_path + 'olist_customers_dataset.csv')
geolocation          = pd.read_csv(data_path + 'olist_geolocation_dataset.csv')
order_items          = pd.read_csv(data_path + 'olist_order_items_dataset.csv',
                                   parse_dates=['shipping_limit_date'])
order_payments       = pd.read_csv(data_path + 'olist_order_payments_dataset.csv')
order_reviews        = pd.read_csv(data_path + 'olist_order_reviews_dataset.csv',
                                   parse_dates=['review_creation_date',
                                                'review_answer_timestamp'])
orders               = pd.read_csv(data_path + 'olist_orders_dataset.csv',
                                   parse_dates=order_date_cols)
products             = pd.read_csv(data_path + 'olist_products_dataset.csv')
sellers              = pd.read_csv(data_path + 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(data_path + 'product_category_name_translation.csv')

tables = {'customers': customers, 'geolocation': geolocation, 
          'order_items': order_items, 'order_payments': order_payments, 
          'order_reviews': order_reviews, 'orders': orders, 
          'products': products, 'sellers': sellers, 
          'category_translation': category_translation}
for name, t in tables.items():
    print(f'{name:25s} rows: {t.shape[0]: >8,} cols: {t.shape[1]}')

customers                 rows:   99,441 cols: 5
geolocation               rows: 1,000,163 cols: 5
order_items               rows:  112,650 cols: 7
order_payments            rows:  103,886 cols: 5
order_reviews             rows:   99,224 cols: 7
orders                    rows:   99,441 cols: 8
products                  rows:   32,951 cols: 9
sellers                   rows:    3,095 cols: 4
category_translation      rows:       71 cols: 2


In [20]:
print('Orders preview:')
display(orders.head())
print('\nDate range of purchases:')
print(orders['order_purchase_timestamp'].min(), '->', orders['order_purchase_timestamp'].max())
print('\nOrder status distribution:')
print(orders['order_status'].value_counts(normalize=True).round(3))

Orders preview:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26



Date range of purchases:
2016-09-04 21:15:19 -> 2018-10-17 17:30:18

Order status distribution:
order_status
delivered      0.970
shipped        0.011
canceled       0.006
unavailable    0.006
invoiced       0.003
processing     0.003
created        0.000
approved       0.000
Name: proportion, dtype: float64


In [21]:
print('=' * 70)
print('MISSING VALUES (only showing columns with nulls)')
print('=' * 70)

for name, t in tables.items():
    nulls = t.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f'\n{name}:')
        for col, n in nulls.items():
            print(f' {col:35s} {n:>7,} ({n/len(t)*100:.1f}%)')

print('\n' + '=' * 70)
print('DUPLICATE ROW CHECK')
print('='*70)
for name, t in tables.items():
    dupes = t.duplicated().sum()
    print(f'{name:25s} duplicates: {dupes:,}')

MISSING VALUES (only showing columns with nulls)

order_reviews:
 review_comment_title                 87,656 (88.3%)
 review_comment_message               58,247 (58.7%)

orders:
 order_approved_at                       160 (0.2%)
 order_delivered_carrier_date          1,783 (1.8%)
 order_delivered_customer_date         2,965 (3.0%)

products:
 product_category_name                   610 (1.9%)
 product_name_lenght                     610 (1.9%)
 product_description_lenght              610 (1.9%)
 product_photos_qty                      610 (1.9%)
 product_weight_g                          2 (0.0%)
 product_length_cm                         2 (0.0%)
 product_height_cm                         2 (0.0%)
 product_width_cm                          2 (0.0%)

DUPLICATE ROW CHECK
customers                 duplicates: 0
geolocation               duplicates: 261,831
order_items               duplicates: 0
order_payments            duplicates: 0
order_reviews             duplicates: 0
orders    

#### Why we use customer_unique_id, not customer_id

In [22]:
# ---- customer_id is per-order; customer_unique_id is the actual person ----
print('Rows in customers table:', len(customers))
print('Distinct customer_id: ', customers['customer_id'].nunique())
print('Distinct customer_unique_id:', customers['customer_unique_id'].nunique())

# How many orders does each unique customer place?
orders_per_unique = customers.groupby('customer_unique_id')['customer_id'].nunique()
print('\nOrders per unique customer:')
print(f'  1 orders:   {(orders_per_unique ==1).sum(): >6,} customers')
print(f'  2 orders:   {(orders_per_unique ==2).sum(): >6,} customers')
print(f'  3+ orders:  {(orders_per_unique >=3).sum(): >6,} customers')
print(f'  Max orders by one customer: {orders_per_unique.max()}')

repeat_rate = (orders_per_unique > 1).mean() * 100
print(f'\nRepeat purchase rate: {repeat_rate:.2f}%')

Rows in customers table: 99441
Distinct customer_id:  99441
Distinct customer_unique_id: 96096

Orders per unique customer:
  1 orders:   93,099 customers
  2 orders:    2,745 customers
  3+ orders:     252 customers
  Max orders by one customer: 17

Repeat purchase rate: 3.12%


In [23]:
# ---- Payments: one order can have multiple rows (split payments, instalments) ----
print('Payment rows per order:')
print(order_payments.groupby('order_id').size().value_counts().head())

payments_by_order = order_payments.groupby('order_id').agg(
    total_payment_value = ('payment_value', 'sum'),
    payment_installments_max = ('payment_installments', 'max'),
    n_payment_methods = ('payment_type', 'nunique'),
    primary_payment_type = ('payment_type', lambda x: x.value_counts().idxmax())).reset_index()

print('\nReview rows per order:')
print(order_reviews.groupby('order_id').size().value_counts())

reviews_by_order = (order_reviews.sort_values('review_answer_timestamp').drop_duplicates('order_id', keep='last')[['order_id', 'review_score', 'review_creation_date']])

print(f'Reviews deduplicated: {len(reviews_by_order):,} unique orders')

Payment rows per order:
1    96479
2     2382
3      301
4      108
5       52
Name: count, dtype: int64

Review rows per order:
1    98126
2      543
3        4
Name: count, dtype: int64
Reviews deduplicated: 98,673 unique orders


In [24]:
products_named = products.merge(category_translation, on='product_category_name', how='left')
products_named['product_category'] = (products_named['product_category_name_english'].fillna(products_named['product_category_name']).fillna('unknown'))
print(f'Distinct categories: {products_named['product_category'].nunique()}')
print(f'Untranslated rows: {products_named['product_category_name_english'].isnull().sum():,}')

Distinct categories: 74
Untranslated rows: 623


In [25]:
fact_orders = (
    order_items
    .merge(orders, on='order_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(products_named[['product_id', 'product_category']], on='product_id', how='left')
    .merge(sellers, on='seller_id', how='left')
    .merge(payments_by_order, on='order_id', how='left')
    .merge(reviews_by_order, on='order_id', how='left')
)

fact_orders['item_total'] = fact_orders['price'] + fact_orders['freight_value']

fact_orders['delivery_days_total'] = (
    fact_orders['order_delivered_customer_date'] - fact_orders['order_purchase_timestamp']
).dt.days

# Ngeative = livered early; Positive = late vs estimate
fact_orders['delivery_vs_estimate_days'] = (
    fact_orders['order_delivered_customer_date'] - fact_orders['order_estimated_delivery_date']
).dt.days

fact_orders['delivery_bucket'] = pd.cut(
    fact_orders['delivery_vs_estimate_days'], 
    bins=[-np.inf, -1, 0, 3, np.inf],
    labels=['early', 'on_time', 'late_1to3d', 'late_4plus']
)

fact_orders['order_year_month'] = fact_orders['order_purchase_timestamp'].dt.to_period('M')
fact_orders['order_year'] = fact_orders['order_purchase_timestamp'].dt.year

print(f'fact_orders shape: {fact_orders.shape}')
print(f'Distinct orders:   {fact_orders['order_id'].nunique():,}')
print(f'Distinct customers:{fact_orders['customer_unique_id'].nunique():,}')

fact_orders shape: (112650, 34)
Distinct orders:   98,666
Distinct customers:95,420


In [26]:
fact_delivered = fact_orders[fact_orders['order_status'] == 'delivered'].copy()

print(f'All orders:      {len(fact_orders):>8,} rows')
print(f'Delivered only:  {len(fact_delivered):>8,} rows '
      f'({len(fact_delivered)/len(fact_orders)*100:.1f}% kept)')

print('\n---- Sanity Checks ----')
print(f'Date range:            {fact_delivered['order_purchase_timestamp'].min().date()}'
      f'  →  {fact_delivered['order_purchase_timestamp'].max().date()}')
print(f'Total revenue (BRL):   {fact_delivered['item_total'].sum():,.2f}')
print(f'Avergae delivery days: {fact_delivered['delivery_days_total'].mean():.1f}')
print(f'Average review score:  {fact_delivered['review_score'].mean():.2f}')
print(f'On-time delivery %:    {(fact_delivered['delivery_vs_estimate_days'] <=0).mean()*100:.1f}%')

import os
os.makedirs('./data/processed', exist_ok=True)

fact_delivered.to_parquet('./data/processed/fact_orders.parquet', index=False)

fact_delivered.to_csv('./data/processed/fact_orders.csv', index=False)

print('\nSaved to ./data/processed/')

All orders:       112,650 rows
Delivered only:   110,197 rows (97.8% kept)

---- Sanity Checks ----
Date range:            2016-09-15  →  2018-08-29
Total revenue (BRL):   15,419,773.75
Avergae delivery days: 12.0
Average review score:  4.08
On-time delivery %:    93.4%

Saved to ./data/processed/
